# Análisis de cambio semánticos (generar datos para análisis)

In [1]:
# Importar librería
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import pickle
import os
import re
import random as rn
import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import bootstrap
import matplotlib.pyplot as plt
import itertools

from gensim.models import Word2Vec
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
tqdm.pandas()

from ast import literal_eval

In [3]:
# Configurar
load_dotenv() # Cargar las variables de entorno del archivo .env
BASE_DIR =  os.getenv("DIR_BASE")
RESULTADOS_DIR = os.getenv("DIR_DATOS_PROCESADOS") # Acceder a las variables de entorno modelos_swmwosge_1008211100
modelo_dir =  RESULTADOS_DIR+ '/archivos_out/modelos_swmwosge_5051100'
pd.set_option('display.max_colwidth', None)

Se selecciona para analizar el modelo de procrustes

In [4]:
basename = 'NN'
iteracion = 50
tam_vector = 50

In [5]:
# estabilidad_procrustes_iter10_tam50
selected_topics_df = pd.read_csv(
    modelo_dir+'/estabilidad_NN/estabilidad_'+basename+'_iter'+str(iteracion)+'_tam'+str(tam_vector)+'.csv',
    converters={'par_periodo': literal_eval})

selected_topics_df.sort_values('similaridad_semantica', ascending=False)

,iteracion,par_periodo,palabra,similaridad_semantica,cantidad_palabras_comun,topn_vecindad_t1,topn_vecindad_t2
0,16,"(2009, 2014)",radiacion,0,0,"['perjudicial', 'efecto', 'diabetico', 'fondo', 'adolescente', 'apto', 'edad', 'tabaco', 'cancer', 'saludable', 'historia', 'victima']","['espacio', 'desfibrilador', 'leyenda', 'identificacion', 'celiaca', 'automatico', 'exhibicion', 'acceso', 'cualquiera', 'gluten', 'difusion', 'rehabilitacion', 'accidente', 'externo', 'contenido', 'diagnostico']"
2715,29,"(2009, 2014)",exposicion,0,0,"['elaborado', 'leyenda', 'envase', 'tabaco', 'perjudicial', 'gluten', 'consumo', 'publicidad', 'diabetico', 'exhibicion', 'producto', 'identificacion', 'alimenticio', 'apto', 'padecer', 'control', 'educacion', 'garantizar', 'contenido']","['proteccion', 'hijo', 'nacido', 'recien', 'espacio', 'accidente', 'asignacion', 'mes']"
2702,22,"(2014, 2019)",congreso,0,0,"['economico', 'obligacion', 'clinica', 'aplicacion', 'discapacidad', 'zona', 'plazo', 'epidemiologico', 'hospital', 'clinico', 'digital', 'potable', 'relacion', 'grupo', 'cordoba']","['social', 'nacion', 'obra', 'entidad', 'prepagar', 'jubilado', 'situacion', 'modificacion', 'medicina', 'seguro']"
2703,7,"(2014, 2019)",dispongar,0,0,"['jubilado', 'modificacion', 'decreto', 'estudio', 'neonatal', 'agente', 'terapia', 'instituto', 'denominado', 'unico']","['dengue', 'prevenibl', 'virus', 'caracter', 'federal', 'zona', 'realizacion', 'economico']"
2704,7,"(2014, 2019)",fortalecimiento,0,0,"['red', 'fondo', 'especial', 'victima', 'nacion', 'violencia', 'hospital', 'banco', 'tdah', 'licenciado', 'deficit', 'clinica', 'ministerio', 'asistencia', 'hiperactividad', 'alto', 'preventivo']","['ingreso', 'evaluacion', 'economico', 'zona', 'seguro', 'agente', 'comedor', 'argentina', 'desfibrilador', 'termino']"
...,...,...,...,...,...,...,...
65195,28,"(2009, 2014)",provincia,-25,25,"['departamento', 'negro', 'neuquen', 'prorrogable', 'ochenta', 'localidad', 'rio', 'santa', 'desastre', 'economico', 'plazo', 'diverso', 'aires', 'zona', 'termino', 'autonoma', 'ciudad', 'ciento', 'volcan', 'productivo', 'territorio', 'consecuencia', 'afectado', 'inundacion', 'junio', 'emergencia', 'hospital']","['santa', 'zona', 'departamento', 'localidad', 'rio', 'productivo', 'inundacion', 'desastre', 'plazo', 'negro', 'prorrogable', 'ochenta', 'ciudad', 'neuquen', 'aires', 'ciento', 'economico', 'volcan', 'diverso', 'mes', 'termino', 'autonoma', 'junio', 'emergencia', 'consecuencia', 'afectado']"
65196,39,"(2009, 2014)",zona,-25,25,"['desastre', 'plazo', 'prorrogable', 'ochenta', 'volcan', 'departamento', 'aires', 'diverso', 'economico', 'emergencia', 'autonoma', 'negro', 'santa', 'provincia', 'neuquen', 'inundacion', 'localidad', 'productivo', 'termino', 'ciento', 'ciudad', 'rio', 'consecuencia', 'declarar', 'afectado', 'junio', 'dengue']","['productivo', 'desastre', 'departamento', 'plazo', 'ciento', 'ochenta', 'santa', 'prorrogable', 'rio', 'economico', 'localidad', 'neuquen', 'negro', 'provincia', 'volcan', 'diverso', 'inundacion', 'termino', 'situacion', 'aires', 'consecuencia', 'emergencia', 'mes', 'afectado', 'junio', 'declarar', 'ciudad']"
65197,44,"(2009, 2014)",cordoba,-25,25,"['ochenta', 'departamento', 'plazo', 'prorrogable', 'negro', 'desastre', 'neuquen', 'consecuencia', 'economico', 'zona', 'santa', 'provincia', 'ciento', 'localidad', 'rio', 'afectado', 'volcan', 'diverso', 'termino', 'aires', 'productivo', 'inundacion', 'autonoma', 'junio', 'dengue', 'emergencia', 'territorio', 'accidente', 'declarar', 'ciudad']","['productivo', 'departamento', 'zona', 'localidad', 'ciento', 'desastre', 'plazo', 'ochenta', 'diverso', 'negro', 'rio', 'neuquen', 'santa', 'economico', 'prorrogable', 'termino', 'provincia', 'aires', 'consecuencia', 'emergencia', 'volcan', 'inundacion', 'declarar', 'situacion', 'junio', 'autonoma', 'mes']"
65198,33,"(2009, 2014)",cordoba,-25,25,"['ochenta', 'departamento', 'prorrogable', 'plazo', 'desastre', 'negro', 'economico', 'zona', 'neuque

In [6]:
selected_topics_df[selected_topics_df['palabra'].isin(['acompanante'])]

,iteracion,par_periodo,palabra,similaridad_semantica,cantidad_palabras_comun,topn_vecindad_t1,topn_vecindad_t2
271,18,"(2009, 2014)",acompanante,0,0,"['mujer', 'embarazado', 'inciso', 'privado', 'situacion', 'vida', 'modificacion', 'recien', 'prohibir', 'vacunacion', 'hijo', 'nacido', 'ciudadano', 'historia', 'adolescente', 'nina']","['ejercicio', 'regulacion', 'actividad', 'profesional', 'asistido', 'rehabilitacion', 'reproduccion', 'medicinal', 'terapia', 'especialidad', 'tecnica']"
475,34,"(2009, 2014)",acompanante,0,0,"['mujer', 'privado', 'situacion', 'inciso', 'nacido', 'embarazado', 'recien', 'espacio', 'hijo', 'modificacion', 'prohibir', 'adolescente', 'implementar', 'externo', 'vida', 'escolar', 'historia', 'desfibrilador', 'asignacion']","['ejercicio', 'regulacion', 'actividad', 'profesional', 'asistido', 'reproduccion', 'rehabilitacion', 'azar', 'terapia', 'medicinal', 'especialidad']"
538,17,"(2009, 2014)",acompanante,0,0,"['mujer', 'situacion', 'privado', 'embarazado', 'vida', 'inciso', 'desfibrilador', 'externo', 'espacio', 'recien', 'modificacion', 'adolescente', 'nacido', 'ciudadano', 'institucion', 'hijo', 'historia', 'implementar', 'garantizar', 'nina']","['ejercicio', 'actividad', 'regulacion', 'profesional', 'asistido', 'deportivo', 'reproduccion', 'rehabilitacion', 'tecnica', 'medicinal', 'terapia', 'droga']"
606,32,"(2009, 2014)",acompanante,0,0,"['mujer', 'embarazado', 'situacion', 'privado', 'inciso', 'vida', 'implementar', 'recien', 'hijo', 'externo', 'espacio', 'desfibrilador', 'automatico', 'modificacion', 'historia', 'vacunacion', 'nacido', 'prohibir', 'victima', 'asignacion']","['ejercicio', 'profesional', 'regulacion', 'actividad', 'asistido', 'reproduccion', 'rehabilitacion', 'medicinal', 'medicina', 'terapia', 'especialidad', 'nutricion']"
666,2,"(2009, 2014)",acompanante,0,0,"['mujer', 'situacion', 'embarazado', 'privado', 'inciso', 'hijo', 'recien', 'desfibrilador', 'modificacion', 'vida', 'nacido', 'espacio', 'automatico', 'externo', 'asignacion', 'historia', 'adolescente']","['ejercicio', 'profesional', 'actividad', 'asistido', 'regulacion', 'reproduccion', 'terapia', 'rehabilitacion', 'azar', 'tecnica', 'juego', 'deportivo', 'nutricion', 'medicina']"
...,...,...,...,...,...,...,...
54793,18,"(2014, 2019)",acompanante,-8,8,"['equinoterapia', 'terapeutico', 'ejercicio', 'regulacion', 'actividad', 'licenciado', 'preventivo', 'profesional', 'asistido', 'receta', 'rehabilitacion', 'reproduccion', 'medicinal', 'terapia', 'epidemiologico', 'especialidad', 'tecnica']","['ejercicio', 'articulo', 'profesional', 'licenciado', 'terapeutico', 'equinoterapia', 'receta', 'regulacion', 'actividad', 'clinico', 'victima', 'derogacion']"
55201,37,"(2014, 2019)",acompanante,-8,8,"['equinoterapia', 'terapeutico', 'ejercicio', 'profesional', 'regulacion', 'actividad', 'licenciado', 'asistido', 'preventivo', 'reproduccion', 'receta', 'deportivo', 'medicinal', 'nutricion', 'terapia', 'especialidad', 'tecnica', 'rehabilitacion']","['articulo', 'ejercicio', 'terapeutico', 'licenciado', 'profesional', 'asistido', 'actividad', 'receta', 'equinoterapia', 'paciente', 'victima']"
55725,12,"(2014, 2019)",acompanante,-8,8,"['equinoterapia', 'terapeutico', 'ejercicio', 'regulacion', 'actividad', 'licenciado', 'asistido', 'receta', 'profesional', 'reproduccion', 'preventivo', 'rehabilitacion', 'tecnica', 'terapia', 'deportivo', 'epidemiologico']","['articulo', 'terapeutico', 'ejercicio', 'profesional', 'licenciado', 'asistido', 'equinoterapia', 'receta', 'victima', 'terapia', 'utilizacion']"
55828,9,"(2014, 2019)",acompanante,-8,8,"['equinoterapia', 'terapeutico', 'ejercicio', 'profesional', 'regulacion', 'actividad', 'licenciado', 'receta', 'asistido', 'preventivo', 'reproduccion', 'rehabilitacion', 'medicinal', 'deportivo', 'medicina', 'epidemiologico']","['terapeutico', 'articulo', 'licenciado', 'ejercicio', 'equinoterapia', 'educacion', 'asistido', 'actividad', 'donante', 'historia', 'red', 'certificado', 'receta', 'prof

In [5]:
selected_topics_df.describe() # Estadística

,iteracion,similaridad_semantica,cantidad_palabras_comun
count,65200.00000,65200.000000,65200.000000
mean,24.50000,-5.244340,5.244340
std,14.43098,3.891113,3.891113
min,0.00000,-25.000000,0.000000
25%,12.00000,-7.000000,3.000000
50%,24.50000,-5.000000,5.000000
75%,37.00000,-3.000000,7.000000
max,49.00000,0.000000,25.000000


Se realizaron 50 ejecuciones independientes del enfoque NN, cada una con una semilla aleatoria distinta. En cada iteración, se seleccionaron las 250 palabras con mayor cambio semántico entre los pares de períodos comparados, considerando únicamente aquellas con al menos 5 ocurrencias en alguno de los dos períodos de cinco años.

Posteriormente, se calculó la similaridad semantica promedio (medida de estabilidad del modelo) para cada palabra identificada en cada par de periodos comparados. Este valor fue estimado junto con sus respectivos intervalos de confianza del 95%, utilizando el método bootstrap.

In [6]:
n = 250
at_least_in_any_decade = 5
pares_periodo = [ (2009, 2014),(2014, 2019)]
topn_periodo = {}

for par_periodo in pares_periodo:
    df =  selected_topics_df[
        selected_topics_df['par_periodo'] == par_periodo
    ].copy()
    
    
    frecPer1 = pd.read_csv(RESULTADOS_DIR+'/archivos_out/frec_para_datos_limpios_por_desplaz_semantico_anios5_'+str(par_periodo[0])+'.csv')
    frecPer2 = pd.read_csv(RESULTADOS_DIR+'/archivos_out/frec_para_datos_limpios_por_desplaz_semantico_anios5_'+str(par_periodo[1])+'.csv')
    frecPer1.columns = ['palabra', 'frec1', 'porc1']
    frecPer2.columns = ['palabra', 'frec2', 'porc2']
    #print(frecPer2.shape)
    topn_all_df = []

    for iter in range(iteracion):
        #print(iter)
        df_topn = df[(df.iteracion==iter)]
        #controlar frec
        #display(df_topn)
        df_topn = df_topn.merge(frecPer1, how='left', on='palabra')
        df_topn = df_topn.merge(frecPer2, how='left', on='palabra')
        df_topn['max_freq_of_any_decade'] = df_topn[['frec1', 'frec2']].max(axis=1)
        df_topn.drop(['frec1', 'frec2', 'porc1', 'porc2'], axis=1, inplace=True) 
        df_topn = df_topn.loc[df_topn.max_freq_of_any_decade>= at_least_in_any_decade].head(n)
        df_topn = df_topn.sort_values('similaridad_semantica', ascending=False).head(n)
        #display(df_topn)
        topn_all_df.append(df_topn)
    
    topn_periodo[par_periodo] = pd.concat(topn_all_df, ignore_index=True)    


#### Intersección promedio
Por el enfoque Procruste para cada par de periodos de 5 años:
¿Cuál es la similaridad semantica promedio (medida de estabilidad del modelo) de los top 250 palabras de las 50 ejecuciones para cada par de periodos comparados con intervalos de confianza del 95% calculados mediante el método bootstrap? 


In [7]:
#def confidence_intervals(data):
#    res = bootstrap((data,), np.mean, confidence_level=0.95)
#    return (res.confidence_interval.low, res.confidence_interval.high)
def confidence_intervals(data):
    """
    Calcula intervalo de confianza con bootstrap de forma robusta
    """
    # Convertir a array numpy y limpiar NaN
    data_clean = np.array(data).copy()
    data_clean = data_clean[~np.isnan(data_clean)]
    
    # Casos especiales donde bootstrap no puede calcularse
    if len(data_clean) < 2:
        # Si hay solo un dato, no hay intervalo de confianza
        if len(data_clean) == 1:
            return (data_clean[0], data_clean[0])
        else:
            return (np.nan, np.nan)
    
    # Si todos los valores son iguales, bootstrap falla
    if np.all(data_clean == data_clean[0]):
        return (data_clean[0], data_clean[0])
    
    try:
        # Configurar bootstrap para evitar problemas con muestras pequeñas
        res = bootstrap(
            (data_clean,), 
            np.mean, 
            confidence_level=0.95,
            random_state=42,  # Para reproducibilidad
            n_resamples=min(1000, len(data_clean) * 100)  # Adaptar a tamaño de muestra
        )
        return (res.confidence_interval.low, res.confidence_interval.high)
    
    except (ValueError, RuntimeError):
        # Fallback: intervalo aproximado usando distribución t
        from scipy import stats
        n = len(data_clean)
        mean = np.mean(data_clean)
        std_err = stats.sem(data_clean)
        
        if n <= 1 or std_err == 0:
            return (mean, mean)
        
        h = std_err * stats.t.ppf((1 + 0.95) / 2., n - 1)
        return (mean - h, mean + h)

In [8]:
topn_periodo_all = topn_periodo.copy() # copiar

In [9]:
#df = topn_periodo[(2009, 2014)].copy()
#data = df['palabra'].value_counts().reset_index()
#data = data[data['count']>=2]
#pal_lista = list(data['palabra'].unique())
#df = df[df['palabra'].isin(pal_lista)]

# Convertir a listas si es necesario
#df['vt1'] = df['topn_vecindad_t1'].apply(lambda x: eval(x) if isinstance(x, str) else x)
#df['vt2'] = df['topn_vecindad_t2'].apply(lambda x: eval(x) if isinstance(x, str) else x)

# Agrupación simplificada
#def custom_agg(group):
#    return pd.Series({
#        'similaridad_mean': group['similaridad_semantica'].mean(),
#        'similaridad_ci': confidence_intervals(group['similaridad_semantica']),
#        'vt1_intersection': set.intersection(*map(set, group['vt1'])) if len(group) > 0 else set(),
#        'vt2_intersection': set.intersection(*map(set, group['vt2'])) if len(group) > 0 else set(),
#        'vt1_union': set.union(*map(set, group['vt1'])) if len(group) > 0 else set(),
#        'vt2_union': set.union(*map(set, group['vt2'])) if len(group) > 0 else set()
#    })

#df = df.groupby(['par_periodo', 'palabra']).apply(custom_agg).reset_index()


In [10]:
df.head(2)

,iteracion,par_periodo,palabra,similaridad_semantica,cantidad_palabras_comun,topn_vecindad_t1,topn_vecindad_t2
2,17,"(2014, 2019)",exhibicion,0,0,"['tabaco', 'elaborado', 'cualquiera', 'distribucion', 'importacion', 'espacio', 'venta', 'expendio', 'publicitario', 'carbono', 'responsable', 'monoxido', 'gastronomico', 'difusion']","['desfibrilador', 'celiaco', 'obligatoriedad', 'ingreso', 'demas', 'estudio']"
3,33,"(2014, 2019)",adecuado,0,0,"['especialidad', 'medicinal', 'alimentario', 'codigo', 'potable', 'nutricional', 'braille', 'seguridad', 'alimentacion', 'producto']","['agente', 'situacion']"


In [ ]:
#df = topn_periodo[(2009, 2014)].copy()
#data = df['palabra'].value_counts().reset_index()
#data = data[data['count']<2]
#pal_lista = list(data['palabra'].unique())
#pal_lista
# (2009, 2014) = ['tratamiento',  'provincia',  'desastre',  'celula', 'pmo', 'ministerio', 'zona', 'ambito']
#(2014, 2019) = ['nina', 'persona', 'medicina', 'potable', 'ley']

In [11]:
for par_periodo in pares_periodo: 
    display(topn_periodo[par_periodo].shape)
    df = topn_periodo[par_periodo].copy()
    data = df['palabra'].value_counts().reset_index()
    data = data[data['count']>=2]
    pal_lista = list(data['palabra'].unique())
    df = df[df['palabra'].isin(pal_lista)]

    # Convertir a listas si es necesario
    df['vt1'] = df['topn_vecindad_t1'].apply(lambda x: eval(x) if isinstance(x, str) else x)
    df['vt2'] = df['topn_vecindad_t2'].apply(lambda x: eval(x) if isinstance(x, str) else x)

    # Agrupación simplificada
    def custom_agg(group):
        return pd.Series({
            'similaridad_semantica_mean': group['similaridad_semantica'].mean(),
            'similaridad_ci': confidence_intervals(group['similaridad_semantica']),
            'vt1': set.intersection(*map(set, group['vt1'])) if len(group) > 0 else set(),
            'vt2': set.intersection(*map(set, group['vt2'])) if len(group) > 0 else set(),
            'vt1_union': set.union(*map(set, group['vt1'])) if len(group) > 0 else set(),
            'vt2_union': set.union(*map(set, group['vt2'])) if len(group) > 0 else set()
        })

    df = df.groupby(['par_periodo', 'palabra']).apply(custom_agg).reset_index()
    topn_periodo[par_periodo] = df

    
  
  


(12500, 8)

C:\Users\Usuario\AppData\Local\Temp\ipykernel_15480\1069459452.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(['par_periodo', 'palabra']).apply(custom_agg).reset_index()


(12500, 8)

C:\Users\Usuario\AppData\Local\Temp\ipykernel_15480\1069459452.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(['par_periodo', 'palabra']).apply(custom_agg).reset_index()


In [12]:
topn_periodo[par_periodo].head(3)

,par_periodo,palabra,similaridad_semantica_mean,similaridad_ci,vt1,vt2,vt1_union,vt2_union
0,"(2014, 2019)",academia,-0.240000,"(-0.38, -0.14)","{preventivo, deportivo, institucion, externo, desfibrilador, centro}","{cordoba, decreto, medica}","{potable, personal, establecimiento, centro, oncologico, espacio, practica, gastronomico, provision, reproduccion, deportivo, educativo, equinoterapia, evaluacion, automatico, preventivo, obligatoriedad, privado, certificado, equipo, actividad, institucion, externo, desfibrilador, apto}","{zona, vih, educacion, infeccion, diabetes, inclusion, marzo, epidemiologico, medicinal, cordoba, elaborado, clinico, instituir, hiperactividad, tdah, misoprostol, visual, licenciado, reproduccion, equinoterapia, semana, tabaco, articulo, especialidad, derivado, receta, decreto, derogacion, ejercicio, medica}"
1,"(2014, 2019)",accesibilidad,-0.660000,"(-0.88, -0.48)","{espacio, equinoterapia, prepagar}",{gratuito},"{potable, relacion, persona, obra, integral, espacio, medicina, proteccion, prevenibl, beneficio, terapia, universal, codigo, vacunacion, encontrar, asistido, seguridad, asignacion, grupo, endometriosis, reproduccion, publico, equinoterapia, garantizar, historia, donante, marco, automatico, acceso, fondo, ingreso, administracion, braille, rehabilitacion, adolescente, desfibrilador, terapeutico, prepagar, seguro}","{neonatal, respuesta, potable, hepatitis, menor, centro, envase, espacio, elaborado, estudio, prevenibl, argentino, practica, condicion, incorporar, pesquisa, instalacion, producto, gratuito, visual, provision, celiaco, virus, licenciado, agua, comercializacion, recien, equinoterapia, historia, tabaco, patologia, donante, preventivo, alimenticio, braille, certificado, libre, rehabilitacion, desfibrilador, externo, embarazado, banco}"
2,"(2014, 2019)",accidente,-3.860465,"(-4.093023255813954, -3.627906976744186)","{vida, monoxido, rehabilitacion, sustancia, responsable, carbono}","{alergia, monoxido, carbono, campana}","{vih, diabetes, cualquiera, fibromialgia, sustancia, campana, distribucion, oncologico, elaborado, prevenibl, alto, mes, dengue, lucha, utilizacion, riesgo, prevencion, paciente, monoxido, promover, semana, publicitario, concientizacion, responsable, tabaco, carbono, derivado, vida, preventivo, sindrome, victima, rehabilitacion}","{anual, diagnostico, fibromialgia, hepatitis, plan, campana, cronico, control, infantil, termino, marzo, oncologico, argentina, epidemiologico, prevenibl, implementacion, alergia, pesquisa, deteccion, ciudadano, endometriosis, regulatorio, prevencion, violencia, tratamiento, programa, monoxido, semana, publicitario, trombofilia, difusion, evaluacion, carbono, nacido, vida, leche, sindrome, mujer, victima, embarazado, vacunacion}"


### Cambios semánticos  

#### Identificar las palabras con cambio semántico importante y moderado



In [13]:
# GUARDAR OBJETO
with open(modelo_dir+'/top250_periodos_NN_df.pkl', 'wb') as file: 
    pickle.dump(topn_periodo,file)

In [14]:
topk = 10
for par_periodo in pares_periodo:

   
    periodo_df = topn_periodo[par_periodo]
    umbral_fuerte1 = 0.3
    periodo_df1 =  periodo_df.sort_values('similaridad_semantica_mean', ascending=False).head(topk)
    # Graficar Heatmaps
    #display(periodo_df1)
    #plot_heatmap(periodo_df1[['palabra','similaridad_semantica_mean']], "Cambio semántico  (["+str(par_periodo[0])+"-"+str(par_periodo[1])+") → ["+str(par_periodo[1])+"-"+str(par_periodo[1]+5)+")", top=topk)
    print("TOP 10 Palabras:", periodo_df1['palabra'].unique())
    display(periodo_df1) 
   

TOP 10 Palabras: ['acompanante' 'dengue' 'legal' 'precio' 'evaluacion' 'materia' 'periodo'
 'ter' 'premio' 'discapacidad']


,par_periodo,palabra,similaridad_semantica_mean,similaridad_ci,vt1,vt2,vt1_union,vt2_union
2,"(2009, 2014)",acompanante,-0.04,"(-0.13602235003367033, 0.0)","{situacion, vida, mujer, recien, privado, hijo, embarazado, inciso, nacido}","{reproduccion, actividad, asistido, regulacion, profesional, ejercicio}","{azar, espacio, situacion, universal, clinico, implementar, nino, familiar, asignacion, ciudadano, anos, deportivo, recien, garantizar, modificacion, historia, perjudicial, hijo, seguro, derecho, prohibir, inciso, nacido, automatico, nina, vida, escolar, mujer, privado, victima, adolescente, desfibrilador, externo, embarazado, institucion, vacunacion}","{nutricion, reproduccion, deportivo, droga, azar, volcan, medicinal, regulacion, especialidad, medicina, terapia, tecnica, impuesto, obligacion, rehabilitacion, actividad, asistido, juego, profesional, ejercicio}"
74,"(2009, 2014)",dengue,-0.06,"(-0.16, -0.02)","{ciento, territorio, sanitario, afectado, seguimiento, departamento, ochenta}","{lucha, virus, gratuito, campana}","{zona, psoriasis, afectado, volcan, declarar, control, termino, prorrogable, cronico, provincia, integral, santa, inter, padecer, ochenta, argentino, accidente, territorio, pais, sanitario, inundacion, enfermedad, republica, departamento, plazo, ciento, consecuencia, neuquen, aires, seguimiento, politica, negro, evaluacion, economico, victima, emergencia, nutricional, desastre}","{vacuna, ministerio, educacion, octubre, campana, federal, junio, control, materno, accidente, implementar, sexual, lucha, gratuito, ciudadano, prevencion, virus, cancer, efecto, politica, responsable, ambito, vida, administracion, ingreso, nacion, vacunacion}"
173,"(2009, 2014)",legal,-0.10,"(-0.2, -0.04)","{complementario, azar, decreto, actividad, medicina, juego, derogacion, ejercicio}","{medicamento, menor, droga}","{nutricion, potable, azar, rotulo, alimento, distribucion, medicinal, medicina, codigo, impuesto, complementario, reproduccion, deportivo, modificacion, hijo, articulo, especialidad, inciso, automatico, decreto, entidad, privado, donacion, actividad, juego, nutricional, prepagar, derogacion, profesional, ejercicio}","{vacuna, psoriasis, pmo, cualquiera, menor, plan, consumo, cronico, inclusion, distribucion, ano, medico, sustancia, medicinal, padecer, estudio, calidad, medicina, terapia, publicidad, edad, incorporar, asistido, contener, obligatorio, droga, promover, comercializacion, garantizar, perjudicial, efecto, seguro, responsable, prohibir, vida, alimenticio, leche, pensionado, apto, embarazado, prepagar, medicamento, importacion, vacunacion, tipo, produccion}"
241,"(2009, 2014)",precio,-0.26,"(-0.4, -0.16)","{tabaco, medicamento, medicinal, regulacion}","{celiaco, anual, apto, cuidado, contenido, celiaca}","{vacuna, ministerio, azar, octubre, rotulo, materno, distribucion, medicinal, elaborado, calidad, publicidad, identificacion, republica, asistido, nino, desarrollo, humano, reproduccion, droga, comercializacion, seguimiento, efecto, politica, concientizacion, tabaco, investigacion, regulacion, especialidad, evaluacion, promocion, ambito, tecnica, publicar, nina, escolar, leche, saludable, adolescente, alimentacion, medicamento, importacion, seguro, ejercicio, produccion}","{gluten, nutricion, anual, alimentario, diagnostico, rotulo, hepatitis, menor, alimento, venta, cuidado, estudio, calidad, implementacion, elaborado, bebida, codigo, practica, diabetico, leyenda, identificacion, forma, asignacion, producto, contener, obligatorio, celiaca, exhibicion, celiaco, recien, comercializacion, promover, perjudicial, seguimiento, hijo, prohibir, regulacion, especialidad, nacido, alimenticio, leche, apto, embarazado, contenido, incluir}"
118,"(2009, 2014)",evaluacion,-0.28,"(-0.44, -0.14)","{nacion, politica, republica, desarrollo}",{},"{vacuna, ministerio, potable, psoriasis, pmo, diabetes, volcan, hepatitis, termino, federal, prorrogable, infantil, materno, calidad, ochenta, universal, territorio, sanitario,

TOP 10 Palabras: ['adecuado' 'adquirido' 'fortalecimiento' 'internacion' 'domiciliario'
 'esencial' 'ingreso' 'tipo' 'dependencia' 'dispositivo']


,par_periodo,palabra,similaridad_semantica_mean,similaridad_ci,vt1,vt2,vt1_union,vt2_union
5,"(2014, 2019)",adecuado,0.00,"(0, 0)","{codigo, potable, alimentario, alimentacion, seguridad, medicinal, especialidad}",{agente},"{nutricion, potable, alimentario, relacion, alimento, venta, medicinal, elaborado, bebida, codigo, identificacion, seguridad, producto, comedor, regulatorio, especial, paciente, historia, derecho, tabaco, etiquetado, especialidad, receta, saludable, braille, obligacion, alimentacion, nutricional, expendio, ejercicio}","{ministerio, personal, cancer, paciente, diabetes, clinica, historia, trombofilia, ambito, lucha, beneficio, salud, situacion, deficit, vida, preventivo, mujer, entidad, victima, encontrar, hiperactividad, asistencial, embarazado, tdah, agente, rcp, atencion}"
8,"(2014, 2019)",adquirido,-0.04,"(-0.14, 0.0)","{vih, fibromialgia, trombofilia, sindrome}","{importacion, vacuna}","{vih, infeccion, pmo, fibromialgia, hepatitis, cronico, sustancia, cuidado, integral, oncologico, padecer, beneficio, hiperactividad, dengue, tdah, lucha, consecuencia, virus, reproduccion, trombofilia, deficit, vida, preventivo, sindrome, mujer, victima, terapeutico, embarazado}","{vacuna, pmo, cualquiera, distribucion, provincia, cordoba, elaborado, prevenibl, alergia, dengue, misoprostol, utilizacion, destinado, sangre, consecuencia, regulatorio, complementario, reproduccion, cardiopulmonar, equinoterapia, tabaco, carbono, donante, marco, receta, decreto, privado, demas, prepagar, importacion}"
142,"(2014, 2019)",fortalecimiento,-0.06,"(-0.16, -0.02)","{especial, fondo, violencia, nacion, victima, red}","{zona, evaluacion, ingreso, economico}","{ministerio, digital, nutricion, oncologico, argentina, red, salud, alto, hiperactividad, tdah, grupo, atencion, especial, licenciado, cancer, violencia, clinica, historia, ambito, hospital, publicar, deficit, preventivo, fondo, asistencia, vida, nacion, victima, banco}","{zona, digital, obra, termino, prorrogable, infantil, argentina, territorio, sanitario, republica, agente, comedor, social, monoxido, sistema, historia, evaluacion, carbono, marco, fondo, ingreso, economico, receta, braille, entidad, libre, emergencia, desfibrilador, instituto, jubilado, nutricional, seguro}"
177,"(2014, 2019)",internacion,-0.06,"(-0.16, -0.02)","{virus, infeccion, sanitario, efecto, cuidado}","{relacion, familiar}","{respuesta, vih, servicio, infeccion, fibromialgia, establecimiento, cuidado, medicinal, calidad, prevenibl, salud, vacunacion, alto, sanitario, dengue, virus, embarazo, violencia, reproduccion, clinica, garantizar, modificacion, historia, efecto, especialidad, evaluacion, hospital, marco, preventivo, ingreso, braille, entidad, externo, seguro}","{personal, relacion, cualquiera, cuidado, terapia, situacion, practica, condicion, identificacion, asistencial, hiperactividad, familiar, asignacion, tdah, atencion, complementario, denominado, mental, deportivo, paciente, equinoterapia, derecho, hospital, vida, ingreso, entidad, legal, victima, equipo, institucion, terapeutico, prepagar}"
102,"(2014, 2019)",domiciliario,-0.12,"(-0.24, -0.04567106157943726)",{medicamento},"{actividad, educacion}","{respuesta, digital, servicio, fibromialgia, materno, cuidado, provincia, epidemiologico, medicinal, envase, cordoba, calidad, red, terapia, universal, codigo, medica, dengue, asistido, lucha, derogacion, provision, regulatorio, cancer, violencia, clinica, promover, garantizar, modificacion, efecto, investigacion, etiquetado, patologia, evaluacion, hospital, publicar, marco, economico, braille, entidad, legal, instituto, jubilado, prepagar, banco, medicamento, seguro, produccion}","{nutricion, educacion, diabetes, cualquiera, afectado, cuidado, respectivamente, terapia, practica, hiperactividad, familiar, tdah, grupo, especial, complementario, licenciado, equinoterapia, semana, patologia, unico, especialidad, hospital, deficit, fondo, economico, certificado, rehabilitacion, actividad, terapeutic

Palabras comunes entre los pares de períodos

In [15]:
# palabra común en entre períodos
palabra_comun = set(topn_periodo[(2009, 2014)]['palabra'].unique()) & set(topn_periodo[(2014, 2019)]['palabra'].unique())
print("Cantidad de palabras en común:",len(palabra_comun))

Cantidad de palabras en común: 234


Top 10

In [16]:
# Período (2009, 2014)
df = topn_periodo[(2009, 2014)]
df = df[df['palabra'].isin(palabra_comun)]
df = df[['palabra','similaridad_semantica_mean','similaridad_ci','vt1','vt2']]
df.columns = ['palabra','ss_mean_2009_2014','ss_ci_2009_2014','vt1_2009_2014','vt2_2009_2014']
# Período (2014, 2019)
df2 = topn_periodo[(2014, 2019)]
df2 = df2[df2['palabra'].isin(palabra_comun)]
df2 = df2[['palabra','similaridad_semantica_mean','similaridad_ci','vt1','vt2']]
df2.columns = ['palabra','ss_mean_2014_2019','ss_ci_2014_2019','vt1_2014_2019','vt2_2014_2019']
# Unión
df = df.merge(df2, how='inner', on='palabra')
df = df[['palabra', 'ss_mean_2009_2014', 'ss_mean_2014_2019', 'ss_ci_2009_2014','ss_ci_2014_2019','vt1_2009_2014','vt2_2009_2014', 'vt1_2014_2019','vt2_2014_2019']]
df['peso'] = (df['ss_mean_2009_2014'] + df['ss_mean_2014_2019'] )/2
print("Top 10 de palabras comunes con Fuerte cambio semántco entre Período [2009, 2014) - > [2014, 2019)  ")
print("Palabras:",df.sort_values('peso').head(10)['palabra'].unique() )
display(df.sort_values('peso').head(10))


Top 10 de palabras comunes con Fuerte cambio semántco entre Período [2009, 2014) - > [2014, 2019)  
Palabras: ['diagnostico' 'lugar' 'instituir' 'vacuna' 'ano' 'republica' 'programa'
 'paciente' 'braille' 'especial']


,palabra,ss_mean_2009_2014,ss_mean_2014_2019,ss_ci_2009_2014,ss_ci_2014_2019,vt1_2009_2014,vt2_2009_2014,vt1_2014_2019,vt2_2014_2019,peso
57,diagnostico,-6.875000,-4.666667,"(-7.0, -6.375)","(-5.0, -4.0)","{capacitacion, psoriasis, tratamiento, pmo, hepatitis, campana, enfermedad, deteccion, difusion, medica, celiaca}","{capacitacion, tratamiento, psoriasis, mujer, hepatitis, seguimiento, deteccion, integral, difusion}","{deficit, alergia, capacitacion, vih, tratamiento, mujer, hepatitis, demas, deteccion, trombofilia, integral, epidemiologico, difusion, embarazado}","{alergia, tratamiento, sindrome, fibromialgia, investigacion, epidemiologico, oncologico}",-5.770833
122,lugar,-6.500000,-5.000000,"(-7.0, -6.0)","(-5, -5)","{leyenda, publico, agua, perjudicial, consumo, desfibrilador, externo, apto, envase, espacio, acceso, gratuito, bebida, exhibicion}","{automatico, perjudicial, desfibrilador, externo, tabaco, espacio, exhibicion}","{automatico, desfibrilador, externo, expendio, espacio}","{automatico, privado, demas, desfibrilador, externo, venta, instalacion}",-5.750000
114,instituir,-6.500000,-4.833333,"(-6.833333333333333, -6.166666666666667)","(-4.958333333333333, -4.625)","{ciudad, escolar, cancer, autonoma, droga, pensionado, octubre, mes, concientizacion, junio, infantil, respectivamente, ano, departamento, jubilado, mundial, argentino}","{diabetico, mujer, pensionado, octubre, mes, junio, ano, jubilado, lucha, sangre}","{cardiopulmonar, semana, mes, ano, jubilado, marzo, rcp, sangre, reanimacion}","{cancer, afectado, semana, mes, ano, marzo, lucha}",-5.666667
227,vacuna,-6.222222,-4.833333,"(-6.555555555555555, -5.777777777777778)","(-5.0, -4.333333333333333)","{virus, leche, administracion, psoriasis, anual, evaluacion, hepatitis, distribucion, padecer, vacunacion, aplicacion, produccion}","{administracion, leche, cualquiera, dengue, distribucion, padecer, vacunacion, tipo, contener}","{alergia, leche, administracion, declarese, cualquiera, dengue, distribucion, padecer, vacunacion, gratuito}","{alergia, virus, prorrogable, dengue, provincia, difusion, vacunacion, prevenibl}",-5.527778
9,ano,-6.000000,-5.000000,"(-6.3076923076923075, -5.6923076923076925)","(-5, -5)","{sexo, instituir, octubre, volcan, movil, mes, junio, departamento, plazo}","{instituir, octubre, menor, mes, junio}","{instituir, mes, semana, marzo, mundial}","{instituir, menor, termino, prorrogable, semana, mes, marzo}",-5.500000
199,republica,-6.333333,-4.617647,"(-6.666666666666667, -6.111111111111111)","(-4.794117647058823, -4.382352941176471)","{educacion, territorio, economico, octubre, promover, desastre, volcan, prorrogable, politica, dengue, plazo, evaluacion, ochenta, argentino}","{territorio, neuquen, obligacion, volcan, inundacion, mes, prorrogable, junio, victima, negro, instalacion, argentino}","{territorio, monoxido, victima, prorrogable, argentina, carbono, gastronomico, argentino}","{desfibrilador, instalacion, argentina, evaluacion, comedor, ciudadano, argentino}",-5.475490
180,programa,-6.187500,-4.750000,"(-6.5625, -5.8125)","(-4.916666666666667, -4.416666666666667)","{ministerio, psoriasis, nacion, pmo, hepatitis, evaluacion}","{pmo, creacion, psoriasis}","{pmo, trombofilia}","{nacional, obligatorio, plan, creacion}",-5.468750
156,paciente,-6.166667,-4.750000,"(-6.666666666666667, -5.5)","(-4.916666666666667, -4.416666666666667)","{ciudad, clinico, denominado, autonoma, diabetico, afectado, seguimiento, historia, junio, efecto, institucion, profesional, terapia}","{clinico, tratamiento, diabetico, afectado, victima, seguimiento, historia, rehabilitacion}","{clinico, tratamiento, alto, relacion, afectado, victima, historia, rehabilitacion, oncologico}","{victima, institucion, oncologico, donante, red}",-5.458333
24,braille,-5.896552,-5.000000,"(-6.103448275862069, -5.724137931034483)","(-5, -5)","{codigo, gluten, droga, identificacion, rotulo, sistema, historia, alimento, efecto, medicinal, especialidad, contener}","{codigo, ident